In [10]:
from pathlib import Path
import sys

import chromadb

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().resolve().parents] if (path / "pyproject.toml").exists()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(repo_root))

from app.config import CHROMA_HOST, CHROMA_PORT, CHROMA_SSL, COLLECTION_NAME
from app.factory import get_embeddings

client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT, ssl=CHROMA_SSL)
collection = client.get_collection(COLLECTION_NAME)
embeddings = get_embeddings()

In [11]:
# Peek at stored documents
peek_results = collection.peek(5)
peek_results

{'ids': ['ebc6d28f41fb9f929a7bf15e7fd7a2b2',
  '0c9bcaf55d902965cd743ab7686ec811',
  'f49233c25593c34a371b6efd3ea8ff6d',
  '0d95c22068939f36e9ceb39be45eec0e',
  '5a630a821e7df9f1feb12d7ff1206b8d'],
 'embeddings': array([[-0.02685251,  0.10350263, -0.16017944, ..., -0.06764706,
         -0.02867484, -0.01133598],
        [ 0.00095201,  0.09941132, -0.16002853, ..., -0.0664585 ,
         -0.00787961,  0.00488184],
        [-0.0024467 ,  0.08375391, -0.1729205 , ..., -0.05865596,
         -0.04051889, -0.01436839],
        [ 0.04353201,  0.04992549, -0.11752061, ..., -0.0268539 ,
         -0.04828065, -0.03143311],
        [-0.00606629,  0.06036356, -0.17388763, ..., -0.0595436 ,
         -0.00974456,  0.00141294]], shape=(5, 768)),
 'metadatas': [{'format': 'md',
   'description': '',
   'source_file': 'knowledge_ingestion/content/v3/content/original paper/Gamage et al. - 2025 - Enhancing Quantum Software Development Process with Experiment Tracking.md',
   'title': 'Gamage et al. - 2025

In [12]:
# Query exactly like your RAG retriever does — see what it returns
query = "NISQ?"
query_embedding = embeddings.embed_query(query)
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {dist:.4f} | Source: {meta}")
    print(doc[:300])
    print("---")

Distance: 0.8950 | Source: {'title': 'Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development', 'source_file': 'knowledge_ingestion/content/v3/content/original paper/Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development.md', 'source_corpus': 'unknown', 'format': 'md', 'description': '', 'content_type': 'narrative'}
_**Index Terms**_ **—Quantum software engineering, development workflow, methodology** 

## I. INTRODUCTION 

The field of quantum computing has evolved rapidly over the past decade, fostered by improvements in quantum computer hardware. These improvements allow researchers to develop increasingly c
---
Distance: 0.9661 | Source: {'source_file': 'knowledge_ingestion/content/v3/content/original paper/Preskill - 2018 - Quantum Computing in the NISQ era and beyond.md', 'source_corpus': 'unknown', 'title': 'Preskill - 2018 - Quantum Computing in the NISQ era and beyond', 'description': '', 

In [13]:
results

{'ids': [['b004bf88233e29d74211dc115a52cb43',
   '3b9c956802dc4e5753c1ce1786f11b30',
   '67fb3e485e9fc640539218e0bf441777',
   '95374698411e10d20896ec7ade2ab53f',
   '0d95c22068939f36e9ceb39be45eec0e']],
 'distances': [[0.894964, 0.966102, 0.9671003, 0.98639226, 1.0184997]],
 'embeddings': None,
 'metadatas': [[{'title': 'Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development',
    'source_file': 'knowledge_ingestion/content/v3/content/original paper/Kinanen et al. - 2025 - Toolchain for Experiment Tracking in Iterative Quantum Software Development.md',
    'source_corpus': 'unknown',
    'format': 'md',
    'description': '',
    'content_type': 'narrative'},
   {'source_file': 'knowledge_ingestion/content/v3/content/original paper/Preskill - 2018 - Quantum Computing in the NISQ era and beyond.md',
    'source_corpus': 'unknown',
    'title': 'Preskill - 2018 - Quantum Computing in the NISQ era and beyond',
    'description': '',
    'conte